In [1]:
!apt-get update
!apt-get install -y openmpi-bin libopenmpi-dev

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,160 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/multiverse amd64 Packages [70.9 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,615

In [2]:
%%writefile hello_mpi.c
#include <mpi.h>
#include <stdio.h>
int main(int argc, char** argv) {
MPI_Init(&argc, &argv);
int rank, size;
MPI_Comm_rank(MPI_COMM_WORLD, &rank);
MPI_Comm_size(MPI_COMM_WORLD, &size);
printf("Hello from process %d of %d\n", rank, size);
MPI_Finalize();
return 0;
}

Writing hello_mpi.c


In [3]:
!mpicc hello_mpi.c -o hello_mpi

In [4]:
!mpirun --allow-run-as-root --oversubscribe -np 4 ./hello_mpi

Hello from process 1 of 4
Hello from process 2 of 4
Hello from process 3 of 4
Hello from process 0 of 4


In [2]:
!nproc

2


In [7]:
#Start of Lab Assignment

In [25]:
%%writefile lab4.c
#include <mpi.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

#define TAG01 100
#define TAG10 101
#define TAGR 200
#define ITERATIONS 10000

double totalTime = 0.0;
double start = 0.0;
double end = 0.0;
int msg_size[] = {1, 8, 64, 512, 4096, 65536};

int run_pingpong(int rank)
{
  char *buffer = malloc(65536);

  for(int m = 0; m < 6; m++)
  {
    totalTime = 0.0;
    for(int i = 0; i < ITERATIONS; i++)
    {
      if(rank == 0)
      {
        start = MPI_Wtime();

        MPI_Send(buffer, msg_size[m], MPI_BYTE, 1, TAG01, MPI_COMM_WORLD);
        MPI_Recv(buffer, msg_size[m], MPI_BYTE, 1, MPI_ANY_TAG, MPI_COMM_WORLD, MPI_STATUS_IGNORE);

        end = MPI_Wtime();
        totalTime += end - start;
      }

      if(rank == 1)
      {
        MPI_Recv(buffer, msg_size[m], MPI_BYTE, 0, MPI_ANY_TAG, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        MPI_Send(buffer, msg_size[m], MPI_BYTE, 0, TAG10, MPI_COMM_WORLD);
      }
    }
    if(rank == 0)
    {
      double RTT = totalTime / ITERATIONS;
      double oneWay = RTT / 2;

      printf("\nmsg_size = %d", msg_size[m]);
      printf("\nRTT: %f\nEstimated One Way: %f\n", RTT, oneWay);
      fflush(stdout);
    }
  }
  free(buffer);
}

void run_ring(int rank, int size)
{
  char *buffer = malloc(65536);
  int next = (rank +1) % size;
  int prev = (rank -1 + size) % size;
  for (int m = 0; m < 6; m++)
  {
    totalTime = 0.0;

    for (int i = 0; i < ITERATIONS; i++)
    {
      MPI_Sendrecv(buffer, msg_size[m], MPI_BYTE, next, TAGR, buffer, msg_size[m], MPI_BYTE, prev, TAGR, MPI_COMM_WORLD, MPI_STATUS_IGNORE);

      printf("\nRank %d: Sending to %d. Receiving from %d.", rank, next, prev);
      fflush(stdout);
    }
  }
  free(buffer);
}

int main(int argc, char** argv)
{
  MPI_Init(&argc, &argv);
  int rank, size;
  MPI_Comm_rank(MPI_COMM_WORLD, &rank);
  MPI_Comm_size(MPI_COMM_WORLD, &size);

  if (argc < 2)
  {
      if (rank == 0)
      printf("Usage: %s pingpong|ring\n", argv[0]);
      MPI_Finalize();
      return 0;
  }

  if (strcmp(argv[1], "pingpong") == 0)
  {
    if (size != 2)
    {if (rank == 0) printf("Ping-pong requires -np 2\n");}
    else
    {run_pingpong(rank);}
  }

  else if (strcmp(argv[1], "ring") == 0)
  {
    if (size < 2)
    {
      if (rank == 0) printf("Ring requires -np >= 2\n");
    }
    else
    {
      run_ring(rank, size);
    }
  }
  MPI_Finalize();
  return 0;
}

Overwriting lab4.c


In [26]:
!mpicc -O3 lab4.c -o lab4

In [27]:
#Run PingPong
!mpirun --allow-run-as-root --oversubscribe -np 2 ./lab4 pingpong


msg_size = 1
RTT: 0.000001
Estimated One Way: 0.000001

msg_size = 8
RTT: 0.000001
Estimated One Way: 0.000001

msg_size = 64
RTT: 0.000001
Estimated One Way: 0.000001

msg_size = 512
RTT: 0.000001
Estimated One Way: 0.000001

msg_size = 4096
RTT: 0.000006
Estimated One Way: 0.000003

msg_size = 65536
RTT: 0.000020
Estimated One Way: 0.000010


In [28]:
#Run Ring
!mpirun --allow-run-as-root --oversubscribe -np 4 ./lab4 ring

Streaming output truncated to the last 5000 lines.
Rank 3: Sending to 0. Receiving from 2.
Rank 3: Sending to 0. Receiving from 2.
Rank 3: Sending to 0. Receiving from 2.
Rank 3: Sending to 0. Receiving from 2.
Rank 3: Sending to 0. Receiving from 2.
Rank 3: Sending to 0. Receiving from 2.
Rank 3: Sending to 0. Receiving from 2.
Rank 3: Sending to 0. Receiving from 2.
Rank 3: Sending to 0. Receiving from 2.
Rank 3: Sending to 0. Receiving from 2.
Rank 3: Sending to 0. Receiving from 2.
Rank 3: Sending to 0. Receiving from 2.
Rank 3: Sending to 0. Receiving from 2.
Rank 3: Sending to 0. Receiving from 2.
Rank 3: Sending to 0. Receiving from 2.
Rank 3: Sending to 0. Receiving from 2.
Rank 3: Sending to 0. Receiving from 2.
Rank 3: Sending to 0. Receiving from 2.
Rank 3: Sending to 0. Receiving from 2.
Rank 3: Sending to 0. Receiving from 2.
Rank 3: Sending to 0. Receiving from 2.
Rank 3: Sending to 0. Receiving from 2.
Rank 1: Sending to 2. Receiving from 0.
Rank 0: Sending to 1. Receivi